<div style="max-width:100%;box-sizing:border-box;overflow:visible;border-top:4px solid #0f766e;padding:32px 0 20px;margin:0 0 24px">
  <div style="display:block;color:#0f766e;font-size:13px;line-height:1.8;font-weight:700;letter-spacing:0.8px;text-transform:uppercase;margin:0 0 8px">LAB 05 · REAL-TIME ANALYTICS WITH APACHE DORIS</div>
  <div style="color:#17212b;font-size:30px;line-height:1.3;font-weight:750;margin:0 0 10px">Explore Doris Functions and Build Analytical Queries</div>
  <p style="color:#475569;font-size:15px;line-height:1.7;max-width:920px;margin:0">Choose a function by the result shape and dependency the business question requires, then turn event-detail rows into grouped metrics, filtered groups, daily-region trends, and a complete analytical result.</p>
  <span style="display:inline-block;border:1px solid #99f6e4;border-radius:4px;background:#f0fdfa;color:#115e59;padding:6px 10px;margin-top:14px;font-size:12px">Scalar · Aggregate · Combinator · Window · Table Function · TVF · AI · CTE · Lambda · UDF family</span>
</div>

This lab continues with the complete `events_modelled` table created in Module 4. The notebook supplies runnable SQL, small literal examples, and interactions so that you can focus on what each function does to its input rows and output rows. The lab does not contact an external AI provider or deploy user code.

### Initialize the Lab

Run the next cell before Section 1. It loads the shared course helper and creates the `lab` object used by every later cell. Run it again after restarting the Jupyter kernel. It does not start Docker or change data in Doris.


In [1]:
from pathlib import Path
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "doris_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from doris_course import DorisLab

lab = DorisLab(lab_dir=COURSE_ROOT);


## 1. Choose a function category from the required result shape

Doris groups functions by the kind of work they perform. Start with the input, output, row-count behavior, and external dependencies instead of memorizing a catalog.

| Function category | Input and output behavior | Representative example |
|---|---|---|
| Scalar function | Produces one value for each input row | `LOWER(event_type)` |
| Aggregate function | Combines values from multiple rows into one value per group | `SUM(revenue)` |
| Combinator | Adapts aggregate behavior for a specialized state or ARRAY task | `SUM_FOREACH(metric_array)` |
| Analytic (window) function | Adds a calculated value while retaining the input result rows | `ROW_NUMBER() OVER (...)` |
| Table Function | Expands a value in each current input row into zero to many rows | `EXPLODE(tags)` with `LATERAL VIEW` |
| Table-Valued Function (TVF) | Supplies a relation that can appear in `FROM` | `NUMBERS(...)` or `S3(...)` |
| AI Function | Calls a model through a configured Doris AI Resource | `AI_SENTIMENT(resource, text)` |

These categories can overlap in one dimension. For example, `AI_SENTIMENT` has scalar row behavior, while the AI category also tells you that the call depends on an external model. An aggregate used with `OVER` follows window semantics and retains its input result rows.

Match each expression to its requirement. The AI example is classified here but is not executed in this lab because it needs an AI Resource, provider access, credentials, and an external model call.

In [2]:
lab.function_category_activity();


<IPython.core.display.Javascript object>

## 2. Distinguish a Table Function from a TVF by where its rows come from

Both categories can return multiple rows, but they enter a query differently.

- A **Table Function** expands a value from each current input row. `EXPLODE(tags)` is used with `LATERAL VIEW`, so the generated rows remain related to `source_id`.
- A **Table-Valued Function (TVF)** is itself a relation in `FROM`. `NUMBERS` generates rows without a source table.

The examples use literals and generated numbers, so they require no object storage or external service. Module 3's `S3` function follows the same TVF relation contract, but this lab does not read S3 again.

In [3]:
lab.connect(container="doris", host="127.0.0.1", port=9030)
lab.execute("USE doris_course")

lab.sql("""
SELECT
    source_id,
    tag
FROM (
    SELECT 1 AS source_id, ['view', 'purchase'] AS tags
) seed
LATERAL VIEW EXPLODE(tags) expanded AS tag
ORDER BY tag
""", title="Table Function: expand one input row")

lab.sql("""
SELECT number
FROM NUMBERS("number" = "4")
ORDER BY number
""", title="TVF: generate a temporary relation");

source_id,tag
1,purchase
1,view


number
0
1
2
3


**Expected result:** the `EXPLODE` query turns the one `seed` row into two rows, `(1, purchase)` and `(1, view)`. Both rows retain `source_id = 1`, showing that they came from that input row. The `NUMBERS` query returns four rows—`0`, `1`, `2`, and `3`—even though no input table is present.

| Operation | Existing input rows | Generated result rows |
|---|---:|---:|
| `EXPLODE(tags)` | 1 row containing a two-element ARRAY | 2 rows tied to that input row |
| `NUMBERS("number" = "4")` | None | 4 rows supplied as a relation |

## 3. Turn stored values into reporting dimensions

Connect to the Frontend (FE), select the course database, and inspect eight matching events. The table keeps the original `event_time` and `region`; scalar functions derive reporting values when the query runs:

- `TO_DATE` converts a timestamp to its calendar date.
- `DATE_TRUNC(..., 'hour')` normalizes timestamps to an hourly boundary.
- `EXTRACT` returns one time component.
- `REPLACE` and `UPPER` produce a presentation label.
- `LIKE` and `REGEXP` select rows by a string pattern.

Each selected input event still produces one output row, so these scalar functions do not change the result grain.


In [9]:
lab.sql("""
SELECT
    event_time,
    TO_DATE(event_time) AS event_date,
    DATE_TRUNC(event_time, 'hour') AS event_hour,
    EXTRACT(HOUR FROM event_time) AS hour_of_day,
    event_type,
    UPPER(REPLACE(region, 'region_', 'REGION-')) AS region_label
FROM events_modelled
WHERE event_time >= '2020-03-03 00:00:00'
  AND event_time <  '2020-03-04 00:00:00'
  AND region LIKE 'region_0%'
  AND event_type REGEXP '^(view|purchase)$'
ORDER BY event_time, event_id
LIMIT 8
""", title="Scalar-function result");


event_time,event_date,event_hour,hour_of_day,event_type,region_label
2020-03-03 00:00:16,2020-03-03,2020-03-03 00:00:00,0,purchase,REGION-01
2020-03-03 00:00:24,2020-03-03,2020-03-03 00:00:00,0,purchase,REGION-07
2020-03-03 00:00:29,2020-03-03,2020-03-03 00:00:00,0,purchase,REGION-05
2020-03-03 00:00:54,2020-03-03,2020-03-03 00:00:00,0,purchase,REGION-03
2020-03-03 00:01:18,2020-03-03,2020-03-03 00:00:00,0,purchase,REGION-01
2020-03-03 00:01:24,2020-03-03,2020-03-03 00:00:00,0,purchase,REGION-01
2020-03-03 00:01:30,2020-03-03,2020-03-03 00:00:00,0,purchase,REGION-08
2020-03-03 00:01:56,2020-03-03,2020-03-03 00:00:00,0,purchase,REGION-02


**Expected result:** eight rows are returned. The first row has `event_date = 2020-03-03`, `event_hour = 2020-03-03 00:00:00`, `hour_of_day = 0`, and `region_label = REGION-01`. Notice that the stored values remain available beside the derived values.


## 4. Reduce detail rows to grouped business metrics

The next query changes the grain from one row per original event to one row per date and region. `COUNT`, `MIN`, `MAX`, and `SUM` are aggregate functions because they combine values across many input rows. For the selected day, 140,782 event rows become eight grouped rows.

The purchase calculations place a conditional expression inside each aggregate function. Non-purchase rows still contribute to `event_count`, but they contribute `NULL` to the purchase `MIN` and `MAX`, and zero to `purchase_revenue`. This produces general activity and purchase-specific measures in the same grouped row.


In [10]:
lab.sql("""
SELECT
    TO_DATE(event_time) AS event_date,
    region,
    COUNT(*) AS event_count,
    MIN(CASE WHEN event_type = 'purchase' THEN revenue END) AS min_purchase_value,
    MAX(CASE WHEN event_type = 'purchase' THEN revenue END) AS max_purchase_value,
    SUM(CASE WHEN event_type = 'purchase' THEN revenue ELSE 0 END) AS purchase_revenue
FROM events_modelled
WHERE event_time >= '2020-03-03 00:00:00'
  AND event_time <  '2020-03-04 00:00:00'
GROUP BY TO_DATE(event_time), region
ORDER BY purchase_revenue DESC, region
""", title="Daily metrics by region");


event_date,region,event_count,min_purchase_value,max_purchase_value,purchase_revenue
2020-03-03,region_02,17838,1.80,2316.64,814438.41
2020-03-03,region_08,17388,1.85,2038.66,803421.62
2020-03-03,region_06,17929,2.57,2205.95,789599.79
2020-03-03,region_01,17724,0.97,2573.81,786295.23
2020-03-03,region_03,17634,2.42,2510.68,784931.15
2020-03-03,region_07,17407,1.03,2110.48,777607.63
2020-03-03,region_04,17175,1.00,2574.04,763191.11
2020-03-03,region_05,17687,2.26,2568.92,727071.45


**Expected result:** eight grouped rows are returned—one for each region on `2020-03-03`. Adding the eight `event_count` values gives **140,782** input events, while adding `purchase_revenue` gives **6,246,556.39**. `min_purchase_value` and `max_purchase_value` describe only purchase rows because aggregate functions ignore the `NULL` produced for other event types.


## 5. Filter input rows with WHERE and aggregated groups with HAVING

This query asks which products generated at least 120,000 in purchase revenue on one day.

| Clause | Operates on | Meaning in this query |
|---|---|---|
| `WHERE` | Original event rows | Only purchase events from the selected day enter aggregation |
| `GROUP BY` | Filtered input rows | One group is created for each `product_id` |
| `HAVING` | Aggregated groups | Only product groups with at least 120,000 in revenue remain |

Moving the revenue threshold into `WHERE` would ask a different question: it would filter individual purchases rather than product totals.


In [ ]:
lab.sql("""
SELECT
    product_id,
    COUNT(*) AS purchase_events,
    SUM(revenue) AS purchase_revenue
FROM events_modelled
WHERE event_time >= '2020-03-03 00:00:00'
  AND event_time <  '2020-03-04 00:00:00'
  AND event_type = 'purchase'
GROUP BY product_id
HAVING SUM(revenue) >= 120000
ORDER BY purchase_revenue DESC, product_id
""", title="Product groups above the revenue threshold");


**Expected result:** six product groups remain after `HAVING`. Product `1005115` ranks first with 593 purchase events and 516,793.33 in revenue; product `1004249` is the last qualifying group with 126,135.40 in revenue.


## 6. Use ANY_VALUE only when any representative value is valid

A grouped query may project a column only when that column defines the group or is calculated by an aggregate function. The first query deliberately violates that rule: it groups by `region` but also requests an unaggregated `event_time` value. Doris rejects the ambiguous projection instead of choosing a timestamp silently.


In [11]:
lab.expected_sql_error("""
SELECT
    region,
    TO_DATE(event_time) AS event_date,
    COUNT(*) AS event_count
FROM events_modelled
WHERE event_time >= '2020-03-03 00:00:00'
  AND event_time <  '2020-03-04 00:00:00'
GROUP BY region
ORDER BY region
""", contains="not in aggregate", title="Doris rejects an ambiguous grouped projection");


The date predicate guarantees that every selected event belongs to the same calendar date. Under that specific condition, choosing any non-`NULL` date from each region is meaningful, so `ANY_VALUE` makes the intention explicit. Do not use it to hide a dimension that should define separate groups.


In [12]:
lab.sql("""
SELECT
    region,
    ANY_VALUE(TO_DATE(event_time)) AS event_date,
    COUNT(*) AS event_count
FROM events_modelled
WHERE event_time >= '2020-03-03 00:00:00'
  AND event_time <  '2020-03-04 00:00:00'
GROUP BY region
ORDER BY region
""", title="A valid representative date for each region");


region,event_date,event_count
region_01,2020-03-03,17724
region_02,2020-03-03,17838
region_03,2020-03-03,17634
region_04,2020-03-03,17175
region_05,2020-03-03,17687
region_06,2020-03-03,17929
region_07,2020-03-03,17407
region_08,2020-03-03,17388


**Expected result:** eight region rows are returned and every `event_date` is `2020-03-03`. Their `event_count` values still sum to 140,782. `ANY_VALUE` is valid here because the date predicate makes the representative date identical across the possible input values.


## 7. Name an aggregated intermediate result with a CTE

A Common Table Expression (CTE) gives a name to an intermediate result within one statement. Here, `daily` names the result produced after event-detail rows have been aggregated to one row per date. It does not create a permanent table.

This first query deliberately stops after the aggregation so that the eight-row input to the next section is visible.


In [13]:
lab.sql("""
WITH daily AS (
    SELECT
        TO_DATE(event_time) AS event_date,
        SUM(CASE WHEN event_type = 'purchase' THEN revenue ELSE 0 END) AS daily_revenue
    FROM events_modelled
    WHERE event_time >= '2020-03-01 00:00:00'
      AND event_time <  '2020-03-09 00:00:00'
    GROUP BY TO_DATE(event_time)
)
SELECT
    event_date,
    daily_revenue
FROM daily
ORDER BY event_date
""", title="Daily aggregate produced by a CTE");


event_date,daily_revenue
2020-03-01,5670241.29
2020-03-02,9933097.98
2020-03-03,6246556.39
2020-03-04,0.00
2020-03-05,0.00
2020-03-06,0.00
2020-03-07,0.00
2020-03-08,0.00


**Expected result:** aggregation produces eight daily rows. Revenue is 5,670,241.29 on `2020-03-01`, 9,933,097.98 on `2020-03-02`, and 6,246,556.39 on `2020-03-03`; the remaining selected dates have zero purchase revenue. These eight rows become the input result rows for the window functions in the next section.


## 8. Add window calculations while retaining the daily rows

The query begins with the same `daily` CTE and therefore the same eight daily rows. Window functions then add calculations to each row:

- `LAG` reads the previous daily result.
- `SUM(...) OVER (...)` produces a running total.
- `ROW_NUMBER` assigns a deterministic revenue rank.

Unlike the earlier aggregate functions, these window functions do not collapse the eight input result rows.


In [14]:
lab.sql("""
WITH daily AS (
    SELECT
        TO_DATE(event_time) AS event_date,
        SUM(CASE WHEN event_type = 'purchase' THEN revenue ELSE 0 END) AS daily_revenue
    FROM events_modelled
    WHERE event_time >= '2020-03-01 00:00:00'
      AND event_time <  '2020-03-09 00:00:00'
    GROUP BY TO_DATE(event_time)
)
SELECT
    event_date,
    daily_revenue,
    LAG(daily_revenue, 1, 0) OVER (ORDER BY event_date) AS previous_revenue,
    SUM(daily_revenue) OVER (
        ORDER BY event_date
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS cumulative_revenue,
    ROW_NUMBER() OVER (
        ORDER BY daily_revenue DESC, event_date
    ) AS revenue_rank
FROM daily
ORDER BY event_date
""", title="Window calculations over the daily rows");


event_date,daily_revenue,previous_revenue,cumulative_revenue,revenue_rank
2020-03-01,5670241.29,0.00,5670241.29,3
2020-03-02,9933097.98,5670241.29,15603339.27,1
2020-03-03,6246556.39,9933097.98,21849895.66,2
2020-03-04,0.00,6246556.39,21849895.66,4
2020-03-05,0.00,0.00,21849895.66,5
2020-03-06,0.00,0.00,21849895.66,6
2020-03-07,0.00,0.00,21849895.66,7
2020-03-08,0.00,0.00,21849895.66,8


**Expected result:** the output still contains the same eight dates produced by the `daily` CTE, but each row now has previous-revenue, cumulative-revenue, and rank values. On `2020-03-02`, the previous revenue is 5,670,241.29 and the cumulative revenue is 15,603,339.27.

| Query stage | Input rows | Output rows | Row behavior |
|---|---:|---:|---|
| Daily aggregation | Event-detail rows | 8 | Rows are collapsed to daily grain |
| Window calculation | 8 daily rows | 8 | Daily rows are retained and calculated columns are added |


## 9. Combine the stages into a daily-region business result

An operations team needs one row per date and region for March 1–3. Each row must show all event activity, distinct active users, distinct purchasing users, purchase revenue, the previous day's revenue for that region, cumulative regional revenue, and the region's revenue rank within the day.

The query separates those responsibilities:

1. `filtered` chooses the input population and derives `event_date`. Its grain remains one row per event.
2. `daily_region` changes the grain to one row per `(event_date, region)` and calculates measures for that group.
3. `scored` applies window functions to the 24 daily-region rows. `PARTITION BY region` defines each region's time series; `PARTITION BY event_date` defines each day's ranking competition.
4. The outer query orders the final rows for display.

A Common Table Expression (CTE) names a result stage within this statement. It does not create a permanent table, and its presence alone does not promise materialization or a performance improvement.

In [4]:
lab.sql("""
WITH filtered AS (
    SELECT
        TO_DATE(event_time) AS event_date,
        region,
        user_id,
        event_type,
        revenue
    FROM events_modelled
    WHERE event_time >= '2020-03-01 00:00:00'
      AND event_time <  '2020-03-04 00:00:00'
),
daily_region AS (
    SELECT
        event_date,
        region,
        COUNT(*) AS event_count,
        COUNT(DISTINCT user_id) AS active_users,
        COUNT(DISTINCT CASE
            WHEN event_type = 'purchase' THEN user_id
        END) AS purchase_users,
        SUM(CASE
            WHEN event_type = 'purchase' THEN revenue ELSE 0
        END) AS purchase_revenue
    FROM filtered
    GROUP BY event_date, region
),
scored AS (
    SELECT
        event_date,
        region,
        event_count,
        active_users,
        purchase_users,
        purchase_revenue,
        LAG(purchase_revenue, 1, 0) OVER (
            PARTITION BY region
            ORDER BY event_date
        ) AS previous_revenue,
        SUM(purchase_revenue) OVER (
            PARTITION BY region
            ORDER BY event_date
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS cumulative_revenue,
        RANK() OVER (
            PARTITION BY event_date
            ORDER BY purchase_revenue DESC
        ) AS revenue_rank
    FROM daily_region
)
SELECT
    event_date,
    region,
    event_count,
    active_users,
    purchase_users,
    purchase_revenue,
    previous_revenue,
    cumulative_revenue,
    revenue_rank
FROM scored
ORDER BY event_date, revenue_rank, region
""", title="Daily-region activity, trend, and rank");

event_date,region,event_count,active_users,purchase_users,purchase_revenue,previous_revenue,cumulative_revenue,revenue_rank
2020-03-01,region_06,9644,4129,1714,748752.94,0.00,748752.94,1
2020-03-01,region_08,9431,4121,1670,739526.04,0.00,739526.04,2
2020-03-01,region_01,9293,4109,1666,732977.08,0.00,732977.08,3
2020-03-01,region_02,9330,3978,1600,713831.18,0.00,713831.18,4
2020-03-01,region_04,9425,4075,1654,704615.19,0.00,704615.19,5
2020-03-01,region_03,9640,4108,1649,679830.92,0.00,679830.92,6
2020-03-01,region_05,9377,4081,1653,675808.09,0.00,675808.09,7
2020-03-01,region_07,9119,4075,1657,674899.85,0.00,674899.85,8
2020-03-02,region_02,13916,6203,2692,1321173.79,713831.18,2035004.97,1
2020-03-02,region_06,14387,6191,2692,1297553.26,748752.94,2046306.20,2


**Expected result:** the query returns **24 rows**: three dates × eight regions. Aggregation establishes the daily-region grain; the window stage retains all 24 rows.

Use `region_02` to read the window columns across time:

- On `2020-03-01`, `purchase_revenue` is 713,831.18, `previous_revenue` is 0.00, and `cumulative_revenue` is 713,831.18.
- On `2020-03-02`, revenue rises to 1,321,173.79. `previous_revenue` is 713,831.18, cumulative revenue becomes 2,035,004.97, and the region ranks first for that date.
- On `2020-03-03`, `previous_revenue` is 1,321,173.79, cumulative revenue becomes 2,849,443.38, and the region again ranks first for that date.

`active_users` and `purchase_users` are distinct within each daily-region group. Do not add these counts across regions or dates and interpret the total as globally distinct users; the same user can occur in more than one group.

| Query stage | Result grain | Observed rows |
|---|---|---:|
| `filtered` | One row per selected event | Detail rows for three dates |
| `daily_region` | One row per date and region | 24 |
| `scored` and final output | One row per date and region | 24 |

## 10. Apply a Lambda expression to each ARRAY element

Doris Lambda expressions commonly appear inside higher-order ARRAY functions. `ARRAY_MAP` transforms each element, while `ARRAY_FILTER` keeps elements that satisfy a condition. This small literal isolates the behavior; it does not change the `events_modelled` schema.


In [ ]:
lab.sql("""
SELECT
    ARRAY_MAP(x -> x * 2, [1, 2, 3, 4]) AS doubled,
    ARRAY_FILTER(x -> x >= 3, [1, 2, 3, 4]) AS kept
""", title="Lambda expressions in ARRAY functions");


**Expected result:** `doubled` is `[2, 4, 6, 8]` and `kept` is `[3, 4]`. The Lambda expression is an argument to a built-in higher-order function; it is not a separately deployed user-defined function (UDF).


## 11. Assemble a business query from bounded choices

Choose a reporting grain, an event filter, and a metric. The notebook generates visible Doris SQL and runs it, so you can connect each choice to its expression without writing an entire statement from an empty editor.

Every selectable operation in this builder uses a Doris built-in function: `TO_DATE` and `DATE_TRUNC` define the reporting grain, while `COUNT`, `COUNT(DISTINCT ...)`, and `SUM` calculate metrics. These functions are already provided by Doris and require no external implementation or registration.

The default choices ask for daily purchase revenue. Change one choice at a time and observe which clause or expression changes and whether the output grain changes.


In [4]:
lab.guided_analysis_builder();


**Expected result:** with the default choices—Day, Purchases only, and Revenue—the leading result is `2020-03-02` with 9,933,097.98, followed by `2020-03-03` with 6,246,556.39 and `2020-03-01` with 5,670,241.29. Selecting Week or Month changes the grouping grain; selecting another metric changes the aggregate expression.

The generated query is composed entirely from built-in functions. The S3 function used in Module 3 is also built in; it belongs to the TVF category because it returns a temporary relation for `FROM`. This lab does not reread the remote dataset because the function-category behavior can be identified without repeating a full object-storage load.


## 12. Recognize when a reusable user-defined function is justified

Doris built-in functions, clear SQL expressions, Lambda expressions, and SQL Alias Functions cover many analytical requirements. A user-defined function family is appropriate when required domain logic is unavailable, must be reused across queries, and justifies packaging, deployment, permissions, and upgrade maintenance.

| Extension | Row behavior | Example requirement |
|---|---|---|
| User-Defined Function (UDF) | One input row produces one value | Apply a proprietary risk score to each customer event |
| User-Defined Aggregate Function (UDAF) | Multiple rows maintain and merge state to produce one value per group | Calculate an industry-specific grouped metric Doris does not provide |
| User-Defined Window Function (UDWF) | A window produces a value for every retained result row | Apply custom window state and reset rules |
| User-Defined Table Function (UDTF) | One input row produces zero to many rows | Parse and expand a proprietary encoded payload with `LATERAL VIEW` |

A Java scalar UDF follows this lifecycle:

```text
Implement an evaluate method
        ↓
Package the implementation as a JAR
        ↓
Make the JAR available to Doris nodes
        ↓
Register its SQL signature with CREATE FUNCTION
        ↓
Call it from SELECT like another scalar function
```

The following example is a registration template, not a runnable cell. It requires a real JAR containing the named Java class and administrative privileges:

```sql
CREATE FUNCTION customer_risk_score(BIGINT, DECIMAL(12, 2))
RETURNS DOUBLE
PROPERTIES (
    "file" = "https://example.org/functions/customer-risk.jar",
    "symbol" = "com.example.analytics.CustomerRiskScore",
    "type" = "JAVA_UDF",
    "always_nullable" = "true",
    "volatility" = "immutable"
);

SELECT
    event_id,
    customer_risk_score(user_id, revenue) AS risk_score
FROM events_modelled;
```

`CREATE FUNCTION` registers the SQL name, input and return types, implementation file, and implementation class; it does not contain the risk algorithm. The Backend (BE) invokes the implementation when the query runs. Mark a function `immutable` only when the same inputs always produce the same output.

**Expected understanding:** the executable analysis in this lab stays on built-in functions. Choose UDF, UDAF, UDWF, or UDTF only when its matching row behavior and reusable domain logic are both required.

### Stop the Doris sandbox

Run this optional cell to release CPU and memory. Lab 5 reads but does not replace `events_modelled`; Docker named volumes preserve the database and table.


In [ ]:
lab.shell(r"""
set -euo pipefail

docker stop doris
docker inspect --format 'container={{.State.Status}}' doris
""", title="Stop the Doris sandbox");


**Expected result:** Docker reports `container=exited`.

### Restart the Doris sandbox

Run this cell before continuing to another module. It starts the existing container, reconnects to FE, and verifies the persisted query-ready model.

This restart cell is idempotent: it can be run when Docker Desktop and the container are stopped, starting, or already running. On macOS it opens Docker Desktop when necessary; on Linux, start Docker Engine before running the cell.


In [ ]:
lab.start_container("doris")

lab.connect(container="doris", host="127.0.0.1", port=9030)
lab.execute("USE doris_course")
lab.sql("SELECT COUNT(*) AS event_rows FROM events_modelled", title="Recovered analytical input");


**Expected result:** the container returns to `healthy`, and `event_rows` is 10,158,080.

## Lab complete

You classified functions by input, output, row-count behavior, and dependency; distinguished a Table Function from a TVF with runnable examples; transformed individual values; reduced detail rows to grouped metrics; separated `WHERE` from `HAVING`; used `ANY_VALUE` only under a valid constraint; named intermediate results with CTEs; observed window functions retain result rows; built a 24-row daily-region analysis; applied Lambda expressions to ARRAY values; assembled a guided analytical query; and selected the appropriate user-defined function family for unsupported reusable logic.

Official references: [SQL Functions](https://doris.apache.org/docs/4.x/sql-manual/sql-functions/) · [Combinators](https://doris.apache.org/docs/4.x/sql-manual/sql-functions/combinators/) · [Table Functions](https://doris.apache.org/docs/4.x/sql-manual/sql-functions/table-functions/) · [Table-Valued Functions](https://doris.apache.org/docs/4.x/sql-manual/sql-functions/table-valued-functions/) · [AI Functions](https://doris.apache.org/docs/4.x/sql-manual/sql-functions/ai-functions/overview/) · [Common Table Expressions](https://doris.apache.org/docs/4.x/query-data/cte/) · [Window Functions](https://doris.apache.org/docs/4.x/query-data/window-function/) · [ANY_VALUE](https://doris.apache.org/docs/4.x/sql-manual/sql-functions/aggregate-functions/any-value/) · [ARRAY_MAP](https://doris.apache.org/docs/4.x/sql-manual/sql-functions/scalar-functions/array-functions/array-map/) · [Java UDF, UDAF, UDWF and UDTF](https://doris.apache.org/docs/4.x/query-data/udf/java-user-defined-function/) · [CREATE FUNCTION](https://doris.apache.org/docs/4.x/sql-manual/sql-statements/function/CREATE-FUNCTION/)